# Etapa 2.1 — Modelo preditivo de conversão

Vamos usar **Regressão Logística**, um modelo simples de classificação, para estimar a chance de cada lead virar cliente. O código usa o arquivo `leads.csv` que veio no projeto.

In [ ]:
# Bibliotecas usadas no projeto
from pathlib import Path
import pandas as pd
import joblib
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

## 1. Carregar e conferir os dados

O CSV fica na pasta principal. O caminho abaixo funciona abrindo o notebook pelo VS Code ou pelo Jupyter.

In [ ]:
# Encontra o CSV mesmo que o notebook seja executado de pastas diferentes
pasta_projeto = Path("..") if Path("../leads.csv").exists() else Path(".")
dados = pd.read_csv(pasta_projeto / "leads.csv")

print("Quantidade de leads:", len(dados))
print("Conversões:")
print(dados["converteu_em_cliente"].value_counts())
print("Dados ausentes:", dados.isna().sum().sum())
dados.head()

## 2. Escolher os dados de entrada e a resposta

`id_visitante` é somente um identificador. A coluna `converteu_em_cliente` é a resposta que queremos prever.

In [ ]:
# Informações que o modelo pode usar para prever
colunas = ["cidade_lead", "setor_empresa", "tempo_navegacao_site", "paginas_visitadas"]
X = dados[colunas]

# 1 = converteu, 0 = não converteu
y = dados["converteu_em_cliente"].map({"sim": 1, "não": 0})

# 80% para aprender e 20% para avaliar com dados não vistos
X_treino, X_teste, y_treino, y_teste = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
print("Leads no treino:", len(X_treino))
print("Leads no teste:", len(X_teste))

## 3. Preparar as colunas e treinar o modelo

Nomes de cidades e setores são transformados em colunas numéricas. Os valores numéricos são colocados em uma escala comparável. O preparo é aprendido **somente com o treino**, para não usar informações do teste por engano.

In [ ]:
# Preparação das informações para a Regressão Logística
preparacao = ColumnTransformer([
    ("categorias", OneHotEncoder(handle_unknown="ignore"), ["cidade_lead", "setor_empresa"]),
    ("numeros", StandardScaler(), ["tempo_navegacao_site", "paginas_visitadas"]),
])

# O Pipeline junta a preparação e o modelo
modelo = Pipeline([
    ("preparacao", preparacao),
    ("classificador", LogisticRegression(max_iter=1000)),
])

modelo.fit(X_treino, y_treino)
print("Modelo treinado!")

## 4. Avaliar o resultado

- **Acurácia:** porcentagem total de acertos.
- **Precisão:** entre os leads previstos como clientes, quantos realmente converteram.
- **Recall:** entre os leads que realmente converteram, quantos o modelo identificou.
- **F1-score:** equilíbrio entre precisão e recall.

In [ ]:
previsoes = modelo.predict(X_teste)
probabilidades = modelo.predict_proba(X_teste)[:, 1]

print(f"Acurácia: {accuracy_score(y_teste, previsoes):.2%}")
print(f"Precisão: {precision_score(y_teste, previsoes):.2%}")
print(f"Recall: {recall_score(y_teste, previsoes):.2%}")
print(f"F1-score: {f1_score(y_teste, previsoes):.2%}")

# Linhas: resultado real (não/sim). Colunas: previsão (não/sim).
print("Matriz de confusão:")
print(confusion_matrix(y_teste, previsoes))

## 5. Exemplo: probabilidade de novos leads

Esses dois exemplos são fictícios, apenas para mostrar como a previsão funciona.

In [ ]:
novos_leads = pd.DataFrame([
    {"cidade_lead": "São Paulo", "setor_empresa": "Tecnologia", "tempo_navegacao_site": 1100, "paginas_visitadas": 13},
    {"cidade_lead": "Recife", "setor_empresa": "Saúde", "tempo_navegacao_site": 350, "paginas_visitadas": 4},
])

novos_leads["chance_de_conversao"] = modelo.predict_proba(novos_leads)[:, 1]
novos_leads["chance_de_conversao"] = novos_leads["chance_de_conversao"].map(lambda valor: f"{valor:.1%}")
novos_leads

## 6. Salvar o modelo

Esse arquivo pode ser usado futuramente por uma API, sem treinar novamente a cada solicitação.

In [ ]:
joblib.dump(modelo, pasta_projeto / "modelo_leads.joblib")
print("Modelo salvo em:", (pasta_projeto / "modelo_leads.joblib").resolve())